# 🤖 OpenAI API로 LLM 다뤄보기

**생성형 AI 개발 · 4차시 실습 — API 기반 생성형 AI 서비스 기초**

---

이 노트북에서는 여러분이 직접 **OpenAI API 키**를 입력하고,
**프롬프트를 작성**해서 LLM의 답변을 받아봅니다.

### 오늘 배우는 것
| 단계 | 내용 |
|---|---|
| 1 | 라이브러리 설치 & API 키 입력 |
| 2 | 첫 요청 보내기 (Hello, GPT!) |
| 3 | 내 프롬프트로 질문하기 |
| 4 | `system` 메시지로 역할 부여 (role prompting) |
| 5 | zero-shot · few-shot 프롬프팅 |
| 6 | `temperature` · `max_tokens` 파라미터 실험 |
| 7 | 멀티턴 대화 만들기 |
| 8 | 실습 과제 |

> 💡 **API 키가 필요합니다.** https://platform.openai.com/api-keys 에서 발급받으세요.
> 키는 `sk-...` 형태이며, **절대 남에게 공유하거나 코드에 그대로 적어두지 마세요.**


## 1단계 · 준비하기

먼저 OpenAI 공식 파이썬 라이브러리를 설치합니다. (이미 설치돼 있으면 그냥 넘어갑니다.)

In [1]:
# OpenAI 라이브러리 설치
%pip install openai --quiet

# 설치 후 정상적으로 불러와지는지 확인
try:
    import openai
    print("✅ 설치 완료 · openai", openai.__version__)
except ImportError:
    print("⚠️ 불러오기 실패 → 상단 메뉴 [런타임] ▸ [런타임 다시 시작] 후 이 셀을 다시 실행하세요.")

✅ 설치 완료 · openai 2.43.0


### API 키 입력

아래 셀을 실행하면 입력창이 뜹니다. 거기에 여러분의 API 키를 붙여넣으세요.
`getpass`를 쓰기 때문에 키가 화면에 **보이지 않게** 안전하게 입력됩니다.

In [2]:
import os
import getpass

# 입력한 키는 화면에 표시되지 않습니다.
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요 (sk-...): ")

print("✅ API 키가 등록되었습니다.")

OpenAI API Key를 입력하세요 (sk-...): ··········
✅ API 키가 등록되었습니다.


### 클라이언트 만들기

`client`는 우리가 OpenAI에게 요청을 보내는 **통로**입니다.
위에서 등록한 환경변수(`OPENAI_API_KEY`)를 자동으로 읽어옵니다.

> ⚠️ **모델 선택 주의**
> - `gpt-4o-mini` · `gpt-4.1-mini` : 일반(비추론형) 모델 → 이 실습 코드가 **그대로 동작**합니다. (추천)
> - `gpt-5.4-mini` · `gpt-5.4-nano` 등 **GPT-5 계열은 추론형**이라 `temperature`를 바꿀 수 없고(1 고정),
>   `max_tokens` 대신 `max_completion_tokens`를 써야 합니다. (아래 `ask()` 함수가 알아서 처리하지만,
>   6단계의 temperature 비교 실습 결과는 달라지지 않습니다.)

In [3]:
from openai import OpenAI

client = OpenAI()   # 환경변수에서 API 키를 자동으로 읽어옵니다

# 사용할 모델 이름 (원하면 아래 중 하나로 바꿔보세요)
#   "gpt-4o-mini"    → 가장 저렴한 일반 모델 (실습에 추천 · 기본값)
#   "gpt-4.1-mini"   → 조금 더 성능이 좋은 일반 모델
#   "gpt-5.4-mini"   → 최신 추론형 모델 (temperature 고정 주의)
#   "gpt-5.4"        → 상위 추론형 모델 (비용 높음)
MODEL = "gpt-4o-mini"

print("✅ 클라이언트 준비 완료 · 사용 모델:", MODEL)

✅ 클라이언트 준비 완료 · 사용 모델: gpt-4o-mini


## 2단계 · 첫 요청 보내기

LLM에게 메시지를 보내는 기본 형태는 다음과 같습니다.

```python
client.chat.completions.create(
    model=MODEL,          # 어떤 모델을 쓸지
    messages=[            # 대화 내용 (list)
        {"role": "user", "content": "질문 내용"}
    ]
)
```

- `messages`는 **대화 목록**입니다. `role`이 `"user"`면 사람, `"assistant"`면 GPT, `"system"`이면 역할 지시입니다.
- 답변 텍스트는 `response.choices[0].message.content` 안에 들어 있습니다.
- 답변 길이(`max_tokens`)·창의성(`temperature`) 같은 옵션은 6단계에서 다룹니다.

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "안녕하세요! 당신을 한 문장으로 소개해 주세요."}
    ]
)

# 답변 텍스트 꺼내기
print(response.choices[0].message.content)

안녕하세요! 저는 다양한 주제에 대한 정보와 도움을 제공하는 인공지능 챗봇입니다.


### 응답 안에는 뭐가 들어있을까?

답변 말고도 **몇 개의 토큰을 썼는지** 같은 정보가 함께 옵니다.
토큰은 대략적인 '단어 조각' 단위이고, 요금이 토큰 수에 따라 매겨지므로 알아두면 좋습니다.

In [5]:
print("입력 토큰 수 :", response.usage.prompt_tokens)
print("출력 토큰 수 :", response.usage.completion_tokens)
print("멈춘 이유    :", response.choices[0].finish_reason)

입력 토큰 수 : 20
출력 토큰 수 : 24
멈춘 이유    : stop


## 3단계 · 내 프롬프트로 질문하기 ✍️

매번 긴 코드를 쓰면 번거로우니, **질문을 넣으면 답변을 돌려주는 함수**로 감싸겠습니다.
이제부터는 `ask("질문")` 한 줄이면 됩니다.

> `system` 인자를 주면 대화 맨 앞에 역할 지시 메시지가 추가됩니다.
> GPT-5 계열 모델도 에러 없이 돌아가도록 파라미터 이름을 자동으로 맞춰 줍니다.

In [8]:
# 메세지 형식
'''
client.chat.completions.create(
    model="gpt-4o-mini",          # ← 최상위 인자
    temperature=1.0,              # ← 최상위 인자
    max_tokens=1024,              # ← 최상위 인자
    messages=[                    # ← 최상위 인자 (리스트)
        {"role": "system", "content": "너는 친절한 튜터야."},   # 원소 1 (딕셔너리)
        {"role": "user",   "content": "재귀가 뭐야?"},          # 원소 2 (딕셔너리)
    ]
)
'''

'\nclient.chat.completions.create(\n    model="gpt-4o-mini",          # ← 최상위 인자\n    temperature=1.0,              # ← 최상위 인자\n    max_tokens=1024,              # ← 최상위 인자\n    messages=[                    # ← 최상위 인자 (리스트)\n        {"role": "system", "content": "너는 친절한 튜터야."},   # 원소 1 (딕셔너리)\n        {"role": "user",   "content": "재귀가 뭐야?"},          # 원소 2 (딕셔너리)\n    ]\n)\n'

In [9]:
def ask(prompt, system=None, max_tokens=1024, temperature=1.0):
    """프롬프트를 보내고 LLM의 답변 텍스트를 돌려줍니다."""
    messages = []
    if system:                                       # 역할(system) 지시가 있으면 맨 앞에 추가
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    kwargs = {"model": MODEL, "messages": messages}
    if MODEL.startswith("gpt-5"):                     # GPT-5 계열(추론형)은 파라미터가 다름
        kwargs["max_completion_tokens"] = max_tokens  #  · max_tokens 대신 이것을 사용
        #  · temperature는 기본값(1)만 지원하므로 넣지 않음
    else:                                            # gpt-4o / gpt-4.1 계열(일반)
        kwargs["max_tokens"] = max_tokens
        kwargs["temperature"] = temperature

    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

print("✅ ask() 함수 준비 완료")

✅ ask() 함수 준비 완료


👇 **여기가 핵심입니다.** 아래 따옴표 안의 프롬프트를 **여러분이 원하는 질문으로 바꿔가며** 실행해 보세요.

In [10]:
my_prompt = "파이썬 리스트와 튜플의 차이를 초보자에게 설명해줘. 예시 코드도 하나 보여줘."

print(ask(my_prompt))

파이썬에서 리스트(list)와 튜플(tuple)은 모두 데이터를 모아놓는 자료형이지만, 몇 가지 중요한 차이가 있습니다. 초보자를 위해 간단하게 설명해드릴게요.

### 1. 변경 가능성 (Mutability)
- **리스트**는 변경 가능합니다. 즉, 한 번 생성한 리스트의 요소를 추가, 삭제, 수정할 수 있습니다.
- **튜플**은 변경 불가능합니다. 즉, 한 번 생성한 튜플의 요소는 수정할 수 없습니다.

### 2. 괄호
- 리스트는 대괄호 `[]`를 사용하여 생성합니다.
- 튜플은 소괄호 `()`를 사용하여 생성합니다.

### 3. 용도
- 리스트는 데이터의 집합을 저장하고 필요에 따라 수정해야 할 때 사용합니다.
- 튜플은 고정된 데이터 집합을 저장할 때 사용하며, 데이터의 무결성을 유지하고자 할 때 적합합니다.

예시 코드를 통해 구체적으로 보여드리겠습니다.

```python
# 리스트 예시
my_list = [1, 2, 3]
print("리스트:", my_list)

# 리스트의 원소 추가
my_list.append(4)
print("원소 추가 후 리스트:", my_list)

# 리스트의 원소 수정
my_list[0] = 100
print("원소 수정 후 리스트:", my_list)

# 튜플 예시
my_tuple = (1, 2, 3)
print("튜플:", my_tuple)

# 튜플의 원소 수정 시도 (주석 처리된 부분은 에러가 발생)
# my_tuple[0] = 100  # TypeError: 'tuple' object does not support item assignment

# 튜플은 고정되어 있으므로, 아래와 같이 새로운 튜플로 재정의해야 함
my_tuple = (100, 2, 3)
print("새로운 튜플:", my_tuple)
```

### 출력 결과:
```
리스트: [1, 2, 3]
원소 추가 후 리스트: [1, 2, 3, 4]
원소 수정 후 리스트: [100, 2, 3, 4]
튜플: (1, 2, 3)
새로운 튜플: (

> 🔁 위 셀의 `my_prompt` 내용을 자유롭게 바꿔서 여러 번 실행해 보세요.
> 예) `"오늘 점심 메뉴 3개만 추천해줘"`, `"재귀함수를 비유로 설명해줘"`, `"이 문장을 영어로 번역해줘: 생성형 AI는 재미있다"`

## 4단계 · 역할 부여하기 (System Message · Role Prompting)

`system` 메시지는 LLM에게 **"너는 이런 역할이야"** 라고 성격·말투·규칙을 정해주는 지시입니다.
같은 질문이라도 역할에 따라 답변이 완전히 달라집니다.

In [11]:
question = "재귀(recursion)가 뭐야?"

print("=== 역할: 엄격한 교수님 ===")
print(ask(question, system="너는 컴퓨터공학과 교수다. 정확한 용어를 사용해 격식 있게 설명한다."))

print("\n=== 역할: 친근한 선배 ===")
print(ask(question, system="너는 다정한 학과 선배다. 반말로 친근하게, 쉬운 비유를 들어 설명한다."))

=== 역할: 엄격한 교수님 ===
재귀(Recursion)는 프로그래밍 및 수학에서 함수 또는 알고리즘이 자기 자신을 호출하는 방식입니다. 일반적으로 재귀는 문제를 더 작은 하위 문제로 분할하여 해결하는 아이디어에 기반하고 있으며, 이러한 방식은 주어진 문제를 해결하기 위해 동일한 문제를 반복적으로 적용하게 됩니다.

재귀 함수는 다음과 같은 두 가지 주요 구성 요소를 가지고 있습니다:

1. **기저 사례(Base Case)**: 재귀 호출이 종료되는 조건입니다. 기저 사례는 함수가 더 이상 자기 자신을 호출하지 않고 직접적인 결과를 반환하는 조건입니다. 이 조건이 없으면 함수는 무한히 자기 자신을 호출하며 프로그레션할 수 없게 됩니다.

2. **재귀 사례(Recursive Case)**: 함수가 자기 자신을 호출하는 조건입니다. 이 경우 문제를 더 간단하게 만들어서 해결할 수 있도록, 일반적으로 입력 값을 감소시키거나 변경하여 다음 호출 시 적절한 기저 사례에 도달할 수 있도록 합니다.

재귀는 주로 계산적 문제를 해결하기 위해 유용하며, 특히 분할 정복(Divide and Conquer), 동적 프로그래밍(Dynamic Programming), 그리고 그래프 탐색(DFS) 등의 알고리즘에서 널리 사용됩니다.

예를 들어, 팩토리얼 계산을 위한 재귀 함수를 정의할 수 있습니다:

```python
def factorial(n):
    if n == 0:  # 기저 사례
        return 1
    else:       # 재귀 사례
        return n * factorial(n - 1)
```

위의 코드에서 `factorial` 함수는 입력값 `n`이 0일 때, 즉 기저 사례에 도달하면 1을 반환합니다. 그렇지 않은 경우에는 자기 자신을 호출하며 `n`을 감소시키면서 계산을 계속합니다. 이 과정을 통해 최종적으로 재귀 호출은 기저 사례에 도달하게 되고, 그 결과는 역으로 반환되며 최종적인 팩토리얼 값을 얻게 됩니다.

=== 역할:

> 🔁 `system` 내용을 바꿔서 **해적 말투**, **초등학생도 알아듣게**, **한 줄 요약만** 등 다양하게 시켜 보세요.

## 5단계 · Zero-shot vs Few-shot 프롬프팅

- **Zero-shot**: 예시 없이 그냥 시키는 것
- **Few-shot**: **예시 몇 개를 보여준 뒤** 같은 방식으로 하라고 시키는 것 → 형식·스타일을 더 잘 따라옵니다.

In [12]:
# Zero-shot : 예시 없이 바로 요청
zero_shot = "다음 문장의 감정을 '긍정' 또는 '부정'으로 답해줘: 이 영화 정말 시간 아까웠어."
print("[Zero-shot]")
print(ask(zero_shot))

[Zero-shot]
부정


In [13]:
# Few-shot : 예시를 먼저 보여주고 같은 형식으로 답하게 하기
few_shot = """다음 예시처럼 문장의 감정을 분류해줘.

문장: 오늘 날씨가 정말 좋아서 기분이 최고야!
감정: 긍정

문장: 버스를 놓쳐서 지각했어. 최악이야.
감정: 부정

문장: 이 식당 음식은 정말 훌륭했어.
감정:"""

print("[Few-shot]")
print(ask(few_shot))

[Few-shot]
긍정


> 💡 Few-shot은 **원하는 출력 형식을 예시로 고정**하고 싶을 때 특히 강력합니다.
> (예: 항상 JSON으로 답하게 하기, 항상 3줄 요약만 하게 하기)

## 6단계 · 파라미터 실험하기

두 가지 자주 쓰는 값을 조절해 봅시다.

| 파라미터 | 의미 | 값이 클 때 | 값이 작을 때 |
|---|---|---|---|
| `temperature` | 답변의 **무작위성/창의성** (0 ~ 2) | 다양·창의적 | 일관·정확 |
| `max_tokens` | 답변의 **최대 길이** | 길게 가능 | 짧게 잘림 |

> ⚠️ 이 실습은 **일반 모델(gpt-4o-mini 등)** 기준입니다.
> GPT-5 계열은 `temperature`가 1로 고정되어 두 결과가 거의 같게 나옵니다.

In [14]:
prompt = "AI를 주제로 짧은 삼행시를 지어줘."

print("=== temperature = 0.0 (거의 항상 비슷) ===")
print(ask(prompt, temperature=0.0))

print("\n=== temperature = 1.0 (다양하게) ===")
print(ask(prompt, temperature=1.0))

=== temperature = 0.0 (거의 항상 비슷) ===
인공지능, 미래를 열어  
지혜의 바다를 헤엄쳐  
능력의 한계를 넘어가네.

=== temperature = 1.0 (다양하게) ===
안녕하세요! 아래와 같이 AI를 주제로 삼행시를 지어보았습니다.

아: 알고리즘의 진화,  
이: 인공지능의 미래,  
가: 가능성을 펼쳐 나가네.


In [15]:
# max_tokens 를 작게 주면 답변이 도중에 잘립니다.
print("=== max_tokens = 30 (짧게 잘림) ===")
print(ask("인공지능의 역사를 설명해줘.", max_tokens=30))

=== max_tokens = 30 (짧게 잘림) ===
인공지능(AI)의 역사는 여러 단계로 나눌 수 있으며, 그 발전은 과학, 수학, 철학, 컴


## 7단계 · 멀티턴 대화 (대화 기억하기)

LLM은 기본적으로 **이전 대화를 기억하지 못합니다.**
기억하게 하려면 지금까지의 대화 전체를 `messages` 리스트에 담아 매번 함께 보내야 합니다.
`user` → `assistant` → `user` → ... 순서로 번갈아 쌓습니다.

In [16]:
# 대화 기록을 담을 리스트
history = []

def chat(user_message):
    """이전 대화를 기억하며 이어서 대화합니다."""
    history.append({"role": "user", "content": user_message})

    kwargs = {"model": MODEL, "messages": history}
    if MODEL.startswith("gpt-5"):                 # GPT-5 계열은 파라미터 이름이 다름
        kwargs["max_completion_tokens"] = 1024
    else:
        kwargs["max_tokens"] = 1024

    response = client.chat.completions.create(**kwargs)
    answer = response.choices[0].message.content
    history.append({"role": "assistant", "content": answer})  # 답변도 기록에 추가
    return answer

print("✅ chat() 함수 준비 완료")

✅ chat() 함수 준비 완료


In [17]:
print(chat("내 이름은 코알라야. 기억해줘."))

안녕하세요, 코알라! 만나서 반가워요. 어떻게 도와드릴까요?


In [18]:
# 앞에서 이름을 알려줬으니, 이번엔 기억하고 있어야 합니다.
print(chat("내 이름이 뭐라고 했지?"))

당신의 이름은 코알라입니다!


> 🔁 `chat("...")` 을 계속 실행하며 대화를 이어가 보세요.
> 대화를 처음부터 다시 시작하려면 아래 셀로 기록을 비웁니다.

In [19]:
history.clear()
print("🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.")

🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.


## 8단계 · 실습 과제 🎯

아래 빈칸을 채워 직접 만들어 보세요. 정답은 하나가 아닙니다!

**과제 1.** `system` 메시지를 사용해 **"항상 이모지를 섞어 답하는 여행 가이드"** 를 만들고,
"부산 여행 코스를 추천해줘" 라고 물어보세요.

**과제 2.** Few-shot 예시를 만들어, 한글 단어를 넣으면 **영어 단어로만** 답하는 번역기를 만들어 보세요.

**과제 3.** `chat()` 함수로 3턴 이상 대화를 이어가며, LLM이 앞 내용을 기억하는지 확인해 보세요.

In [21]:
# 과제 1: 이모지 여행 가이드
guide_system = "너는 밝고 친근한 여행 가이드다. 답변에 관련 이모지를 자연스럽게 섞어 사용한다."

response1 = ask("부산 여행 코스를 추천해줘.", system=guide_system)
print(response1)

부산은 아름다운 바다와 다양한 매력을 가진 도시죠! 🌊✨ 아래는 알찬 부산 여행 코스를 추천해드릴게요.

### 1일 차: 해양과 문화 탐방

- **광안리 해수욕장** 🏖️  
아침에 광안리 해수욕장에서 해변을 산책하며 상쾌한 바다 바람을 느껴보세요. 광안대교의 멋진 경치도 놓치지 마세요!

- **해운대 해수욕장** 🌅  
해운대로 이동해 해변에서 여유로운 오후를 즐기세요. 해운대 시장에서 맛있는 길거리 음식을 즐길 수도 있답니다! 🌮🍢

- **동백섬 & APEC하우스** 🌴  
해운대에서 동백섬까지 산책하면서 아름다운 자연을 만끽하세요. APEC하우스에서 바다를 배경으로 멋진 사진을 찍어보세요!

- **저녁: 해운대 바다전망 레스토랑** 🍽️  
저녁은 해운대 해변 근처의 레스토랑에서 신선한 해산물 요리를 즐겨보세요. 바다를 보면서 맛있는 식사는 잊을 수 없는 경험이 될 거예요!

---

### 2일 차: 문화와 힐링

- **부산 타워 & 용두산 공원** 🗼  
부산 타워에서 도시 전경을 감상한 뒤, 용두산 공원을 산책해보세요. 아름다운 경치와 함께 피크닉을 즐길 수도 있어요!

- **자갈치 시장** 🦐  
부산의 대표 어시장인 자갈치 시장을 방문해 신선한 해산물을 구경하고, 현지 음식도 맛보세요. 회와 조개구이, 어묵은 필수입니다!

- **국제시장** 🛍️  
국제시장에서 쇼핑을 하며 부산의 다양한 상품들을 구경해보세요. 기념품도 장만하고, 길거리 음식도 놓치지 마세요!

- **부산 영도: 태종대** ⛴️  
영도로 이동해 태종대의 자연경관을 즐겨보세요. 도망갈 수 없는 아름다움에 감탄하게 될 거예요.

---

### 3일 차: 힐링과 자연

- **감천문화마을** 🎨  
아침에 감천문화마을을 방문해 독특한 벽화와 아기자기한 집들을 구경하세요. 예쁜 사진을 찍기에 최고의 장소입니다!

- **부산 미포다리** 🌉  
미포다리 근처에서 점심을 즐기고, 세 해변을 감상하며 여유로운 산책을 해보세요.  

- **부산 해양박물관** 🚢  
마지

In [22]:
# 과제 2 · 과제 3 은 아래 빈 셀에서 자유롭게 작성해 보세요.
# 과제 2: Few-shot 영단어 번역기
few_shot_system = """너는 한글 단어가 주어지면 오직 영어 단어로만 답하는 번역기야. 다른 설명은 절대 하지 마.

[예시]
사과 -> apple
바다 -> ocean
학교 -> school
"""

# 테스트 해보기
response2 = ask("자동차", system=few_shot_system)
print(response2) # 정답인 'car'만 출력되는지 확인하세요!

car


In [23]:
# 과제 3: 대화 기억 확인하기 (3턴 이상)

# 1턴: 정보 입력
print("1턴: ", chat("안녕! 내 이름은 김코딩이고, 지금 파이썬 실습을 하고 있어."))

# 2턴: 일상적인 대화
print("2턴: ", chat("오늘 날씨가 참 좋다. 커피 한 잔 마시고 싶네."))

# 3턴: 앞서 말한 정보(맥락) 확인
print("3턴: ", chat("내가 처음에 내 이름이 뭐라고 했지? 그리고 지금 무슨 공부를 하고 있다고 했는지 기억나?"))

1턴:  안녕하세요, 김코딩님! 파이썬 실습 중이시군요. 어떤 부분에서 도움이 필요하신가요? 궁금한 점이나 배워보고 싶은 내용이 있다면 말씀해 주세요!
2턴:  날씨가 좋을 때 커피 한 잔 즐기는 건 정말 좋은 것 같아요! 커피 한 잔과 함께하는 여유로운 시간은 스트레스 해소에도 도움이 되죠. 요즘 좋아하는 커피 스타일이나 레시피가 있으신가요?
3턴:  네, 김코딩님! 처음에 말씀하신 이름은 김코딩이고, 지금 파이썬 실습을 하고 있다고 하셨습니다. 파이썬 공부는 어떤 주제를 다루고 계신가요? 구체적인 도움이 필요하시면 말씀해 주세요!


---

### 🎉 수고하셨습니다!

오늘 배운 것을 정리하면:

1. `client.chat.completions.create(...)` 로 LLM에 요청을 보낸다
2. `messages` 는 `system` / `user` / `assistant` 로 이루어진 **대화 목록**이다
3. `system` 으로 **역할**을, `temperature`·`max_tokens` 로 **답변 스타일과 길이**를 조절한다
4. **few-shot 예시**로 원하는 출력 형식을 유도할 수 있다
5. 대화를 기억시키려면 **이전 기록을 매번 함께 보낸다**

다음 시간에는 이 API를 **FastAPI 백엔드**와 연동해 실제 챗봇 서비스로 확장합니다. 🚀

> ⚠️ **보안 주의:** API 키가 노출되면 즉시 https://platform.openai.com/api-keys 에서 삭제·재발급하세요.
